# 📊 Sub-Task.AI 기반 보고서 및 시각화 자동 생성 Tutorial 

__Tutorial 목적__

실제 산업에서는 여러 데이터들을 분석한 보고서를 작성하고, 비교/검토하는 작업을 많이 수행합니다. 

본 튜토리얼에서는 정형 데이터를 템플릿에 맞게 작성하고 LLM기반 요약을 작성하는 과정을 수행합니다. 


## 💡 목차
> (1) 사용 데이터셋 설명
> 
> (2) 보고서 최종 목표 소개  
>
> (3) Template 기반 레포트 생성 실습
>
> (4) API (Application Programming Interface) 사용방법
>
> (5) API로 LLM 호출 및 보고서 요약 작성 (Gemini)
>
> (6) 함수로 만들어서 리팩토링
> 
> (7) Streamlit 기반 Dashboard 시각화 (Optional)


## (1) 사용 데이터셋 설명  
### 전자부품(배터리팩) 품질보증 AI 데이터셋

데이터 출처: 
중소벤처기업부, Korea AI Manufacturing Platform(KAMP),
전자부품(배터리팩) 품질보증 AI 데이터셋, 스마트제조혁신추진단(수행기관 : ㈜인터엑스/네스트필드㈜), 
2022.12.23, [KAMP URL](https://www.kamp-ai.kr/aidataDetail?DATASET_SEQ=58&page=1)



## (2) 보고서 최종 목표 소개



## (3) Template 기반 레포트 생성 실습 

### (3)-1. Template 변수 교체 실습

In [3]:
from utils.template_tools2 import *

import pandas as pd
from io import BytesIO
import matplotlib.pyplot as plt
import os

from datetime import datetime

# ──────────────────────────────────────────────
# 사용 예시
# ──────────────────────────────────────────────

if __name__ == "__main__":
    # 교체할 키워드 → 새 값 매핑
    replacements = {
        "{{작성자}}": "홍길동",
        "{{작성일}}": "2024-06-01",
        "{{Pack ID}}": "PACK-001",
        "{{평가기간}}": "2024-06-01 to 2024-06-30",

        ## 1. 용량
        "{{용량그래프}}": "용량 그래프 이미지 또는 데이터로 교체",

        ## 2. 용접
        "{{용접그래프}}": "용접 그래프 이미지 또는 데이터로 교체",

        ## 3. 센서
        "{{센서 그래프}}": "센서 그래프 이미지 또는 데이터로 교체"

    }

    replace_keywords_in_docx(
        input_path="Data/Template/battery_report_template.docx",   # 원본 템플릿 경로
        output_path="Result/template_replaced.docx",    # 결과 파일 경로
        replacements=replacements,
    )

[완료] 'Result/template_replaced.docx' 저장 — 총 6개 단락 교체됨


### (3)-2. 변수에 넣을 값 추출 실습

In [21]:
data_list = os.listdir("Data/Tabular/KAMP_Battery/")
data_list = [f for f in data_list if f.endswith(".csv") and "dchg" in f]

In [22]:
print(data_list)

['1000_dchg.csv', '1002_dchg.csv', '1005_dchg.csv', '1004_dchg.csv', '1003_dchg.csv', '1001_dchg.csv']


In [ ]:
# 데이터 불러오기
df = pd.read_csv(f"Data/Tabular/KAMP_Battery/{data_list[0]}")

In [ ]:
## 그래프 생성 함수 
def make_graph_stream(df, title):
    buf = BytesIO()

    plt.figure(figsize=(6, 4))
    if title=="Capacity Graph":
        # 3. 그래프 1: 용량그래프 (RSOCavg, USOCavg, SOH, Power)
        plt.figure(figsize=(10, 4))
        plt.plot(df["DateTime"], df["RSOCavg"], label="RSOCavg")
        plt.plot(df["DateTime"], df["USOCavg"], label="USOCavg")
        plt.plot(df["DateTime"], df["SOH"], label="SOH")
        plt.plot(df["DateTime"], df["Power"], label="Power")
        plt.title("Capacity Graph")
        plt.xlabel("Time")
        plt.ylabel("Value")

    elif title=="Voltage Graph":
        # plt.plot(df["DateTime"], df["Voltage"], label="Voltage")
        plt.plot(df["DateTime"], df["Vmin"], label="Vmin")
        plt.plot(df["DateTime"], df["Vmax"], label="Vmax")
        plt.plot(df["DateTime"], df["DV"], label="DV")
        plt.title("Cell Balance Graph")
        plt.xlabel("Time")
        plt.ylabel("Voltage / mV")
    elif title=="Temperature Graph":
        plt.plot(df["DateTime"], df["Tmin"], label="Tmin")
        plt.plot(df["DateTime"], df["Tmax"], label="Tmax")
        plt.plot(df["DateTime"], df["Tavg"], label="Tavg")
        plt.title("Temperature Graph")
        plt.xlabel("Time")
        plt.ylabel("Temperature")

    plt.title(title)
    plt.tight_layout()
    plt.savefig(buf, format="png", dpi=200, bbox_inches="tight")
    plt.close()

    buf.seek(0)
    return buf

In [ ]:
# 1. DateTime 컬럼 만들기
df["DateTime"] = pd.to_datetime(df["Date"] + " " + df["Time"])

# 2. 그래프 저장 폴더
output_dir = "Result"

# 6. 분석 텍스트 만들기
summary_text = (
    f"본 보고서는 배터리 팩 {df['SerialNumber'].iloc[0]}의 충방전 데이터를 분석한 결과이다. "
    f"평가 기간은 {df['DateTime'].min()}부터 {df['DateTime'].max()}까지이며, "
    f"총 {len(df)}개의 데이터를 사용하였다."
)

capacity_text = (
    f"RSOCavg 범위는 {df['RSOCavg'].min():.2f}% ~ {df['RSOCavg'].max():.2f}%이고, "
    f"USOCavg 범위는 {df['USOCavg'].min():.2f}% ~ {df['USOCavg'].max():.2f}%이다. "
    f"SOH 평균은 {df['SOH'].mean():.2f}%이며, "
    f"Power 범위는 {df['Power'].min():.2f} ~ {df['Power'].max():.2f}이다."
)

welding_text = (
    f"셀 전압 최소값은 {df['Vmin'].min():.3f}V, 최대값은 {df['Vmax'].max():.3f}V이다. "
    f"셀 전압 편차 DV의 평균은 {df['DV'].mean():.2f}mV이고, "
    f"최대값은 {df['DV'].max():.2f}mV이다."
)

sensor_text = (
    f"온도 최소값은 {df['Tmin'].min():.2f}℃, 최대값은 {df['Tmax'].max():.2f}℃이다. "
    f"평균 온도는 {df['Tavg'].mean():.2f}℃로 확인되었다."
)

remark_text = (
    f"전압 범위는 {df['Voltage'].min():.2f}V ~ {df['Voltage'].max():.2f}V, "
    f"전류 범위는 {df['Current'].min():.2f}A ~ {df['Current'].max():.2f}A, "
    f"온도 범위는 {df['Tavg'].min():.2f}℃ ~ {df['Tavg'].max():.2f}℃이다."
)


# 7. 치환 딕셔너리 만들기
replacements = {
    "{{작성자}}": "홍길동",
    "{{작성일}}": pd.Timestamp.now().strftime("%Y-%m-%d"),
    "{{Pack ID}}": str(df["SerialNumber"].iloc[0]),
    "{{평가기간}}": f"{df['DateTime'].min()} ~ {df['DateTime'].max()}",
    "{{AI Model}}": "Gemini-2.5-flash-lite",
    "{{요약문}}": summary_text,

    "{{용량그래프}}": {
        "type": "image",
        "stream": make_graph_stream(df, "Capacity Graph"),
        "width": 5.8,
    },
    "{{용량분석}}": capacity_text,

    "{{용접그래프}}": {
        "type": "image",
        "stream": make_graph_stream(df, "Voltage Graph"),
        "width": 5.8,
    },
    "{{용접분석}}": welding_text,

    "{{센서그래프}}": {
        "type": "image",
        "stream": make_graph_stream(df, "Temperature Graph"),
        "width": 5.8,
    },
    "{{센서분석}}": sensor_text,

    "{{샘플수}}": str(len(df)),
    "{{전압범위}}": f"{df['Voltage'].min():.2f} ~ {df['Voltage'].max():.2f} V",
    "{{전류범위}}": f"{df['Current'].min():.2f} ~ {df['Current'].max():.2f} A",
    "{{온도범위}}": f"{df['Tavg'].min():.2f} ~ {df['Tavg'].max():.2f} ℃",
    "{{비고}}": remark_text,
}


# 8. docx 치환
replace_keywords_in_docx(
    input_path="./Data/Template/battery_report_template.docx",
    output_path="./Result/battery_report_output.docx",
    replacements=replacements,
)

/tmp/ipykernel_32637/1008043068.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["DateTime"] = pd.to_datetime(df["Date"] + " " + df["Time"])


[완료] './Result/battery_report_output.docx' 저장 — 총 17개 단락 교체됨


17

<Figure size 600x400 with 0 Axes>

## (4) API 사용 방법

### 🌐 API란? (Application Programming Interface)

> API는 **웹사이트에 있는 데이터를 자동으로 가져올 수 있게 해주는 도구**이다.  
> 쉽게 말해, 사람이 직접 다운로드하지 않아도 데이터를 요청하면 자동으로 전달받는 방식이다.

#### 🤖 1. Gemini API 발급 방법

##### 📝 발급 절차

1. **Google AI Studio 접속**
   - 👉 https://aistudio.google.com

2. **Google 계정으로 로그인**

3. **API Key 생성**
   - 화면 내 **“Get API Key” 또는 “Create API Key” 버튼 클릭**

4. **API 키 복사**
   - 생성된 키를 복사하여 저장

---


In [7]:
# Gemini APi 호출 함수

# !pip install google-genai
from google import genai
from utils.api import *

def invoke_llm(contents):
    API_KEY = gemini_api_key ## gemini api key 발급받아서 넣기 
    client = genai.Client(api_key=API_KEY)
    response = client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=contents
    )
    return response.text
    

## (5) API로 LLM 호출 및 보고서 요약 작성 (Gemini)

In [8]:
from docx import Document

# .docx 파일 읽기
doc = Document("./Result/battery_report_output.docx")

# 내용 추출하여 문자열로 할당
content = ""
for paragraph in doc.paragraphs:
    content += paragraph.text + "\n"

# content 변수에 전체 텍스트가 저장됨

In [9]:
content

'배터리 충방전 시험 데이터 분석 레포트\n그래프 및 텍스트 자동 치환용 DOCX 템플릿\n\n요약\n1. 용량\n그래프\n분석 내용\n2. 용접\n그래프\n분석 내용\n3. 센서\n그래프\n분석 내용\n4. 원시 데이터 요약\n\n'

In [ ]:
# content 변수를 LLM에 입력하여 분석 결과 생성
# 출력 형태를 json schema 로 고정하여 분석 결과를 구조화된 형태로 받도록 유도
prompt = f"""
    다음 배터리 보고서 내용을 전반적으로 분석하고 주요 인사이트 및 결론을 요약하고, 이 평가 결과의 종합 판정을 Pass/Fail로 결정해주세요: {content}
    출력 형식은 반드시 json 형태로 {{'summary': '...', 'conclusion': 'Pass' or 'Fail'}}
"""
analysis_result = invoke_llm(prompt)
print(analysis_result)

```json
{
  "summary": "배터리 충방전 시험 데이터 분석 보고서에 대한 전반적인 분석 결과, 주요 인사이트 및 결론은 다음과 같습니다.\n\n**1. 용량:**\n*   **그래프 분석:** (그래프 내용에 따라 구체적인 설명 추가 필요. 예: \"그래프는 초기 충방전 사이클에서 용량이 안정적으로 유지되는 것을 보여줍니다. 후반부로 갈수록 약간의 용량 감소가 관찰되나, 허용 오차 범위 내에 있습니다.\")\n*   **분석 내용:** 초기 용량은 설계 사양을 만족하며, 충방전 횟수가 증가함에 따라 예상되는 수준의 용량 감소를 보였습니다.  전반적으로 용량 유지 능력은 양호합니다.\n\n**2. 용접:**\n*   **그래프 분석:** (그래프 내용에 따라 구체적인 설명 추가 필요. 예: \"용접부 저항 변화를 나타내는 그래프는 모든 시험 구간에서 안정적인 값을 유지하고 있음을 보여줍니다. 급격한 변화나 이상 징후는 발견되지 않았습니다.\")\n*   **분석 내용:** 용접부의 전기적 저항은 시험 기간 동안 매우 안정적으로 측정되었습니다. 이는 용접 불량이나 단선과 같은 잠재적인 문제를 시사하는 변화가 없음을 의미합니다.\n\n**3. 센서:**\n*   **그래프 분석:** (그래프 내용에 따라 구체적인 설명 추가 필요. 예: \"온도 및 전압 센서 데이터 그래프는 충방전 과정 중 정상 범위를 벗어나지 않는 안정적인 값을 나타냅니다. 이상 온도 상승이나 전압 급변은 관찰되지 않았습니다.\")\n*   **분석 내용:** 배터리의 온도 센서 및 전압 센서 데이터는 모든 시험 조건에서 정상 작동 범위를 유지했습니다. 과열이나 비정상적인 전압 변동 없이 안전하게 작동함을 확인했습니다.\n\n**4. 원시 데이터 요약:**\n*   (원시 데이터 요약에 대한 간략한 설명 추가 필요. 예: \"원시 데이터 요약 테이블은 각 시험 조건별 평균 용량, 최대/최소 온도, 평균 전압 등의 핵심 지표를 포함하고 있으며, 이는 상기 분석 내용을 뒷받침합니다.\

In [13]:
analysis_result = analysis_result.replace("```json", "").replace("```", "")
analysis_result

'\n{\n  "summary": "배터리 충방전 시험 데이터 분석 보고서에 대한 전반적인 분석 결과, 주요 인사이트 및 결론은 다음과 같습니다.\\n\\n**1. 용량:**\\n*   **그래프 분석:** (그래프 내용에 따라 구체적인 설명 추가 필요. 예: \\"그래프는 초기 충방전 사이클에서 용량이 안정적으로 유지되는 것을 보여줍니다. 후반부로 갈수록 약간의 용량 감소가 관찰되나, 허용 오차 범위 내에 있습니다.\\")\\n*   **분석 내용:** 초기 용량은 설계 사양을 만족하며, 충방전 횟수가 증가함에 따라 예상되는 수준의 용량 감소를 보였습니다.  전반적으로 용량 유지 능력은 양호합니다.\\n\\n**2. 용접:**\\n*   **그래프 분석:** (그래프 내용에 따라 구체적인 설명 추가 필요. 예: \\"용접부 저항 변화를 나타내는 그래프는 모든 시험 구간에서 안정적인 값을 유지하고 있음을 보여줍니다. 급격한 변화나 이상 징후는 발견되지 않았습니다.\\")\\n*   **분석 내용:** 용접부의 전기적 저항은 시험 기간 동안 매우 안정적으로 측정되었습니다. 이는 용접 불량이나 단선과 같은 잠재적인 문제를 시사하는 변화가 없음을 의미합니다.\\n\\n**3. 센서:**\\n*   **그래프 분석:** (그래프 내용에 따라 구체적인 설명 추가 필요. 예: \\"온도 및 전압 센서 데이터 그래프는 충방전 과정 중 정상 범위를 벗어나지 않는 안정적인 값을 나타냅니다. 이상 온도 상승이나 전압 급변은 관찰되지 않았습니다.\\")\\n*   **분석 내용:** 배터리의 온도 센서 및 전압 센서 데이터는 모든 시험 조건에서 정상 작동 범위를 유지했습니다. 과열이나 비정상적인 전압 변동 없이 안전하게 작동함을 확인했습니다.\\n\\n**4. 원시 데이터 요약:**\\n*   (원시 데이터 요약에 대한 간략한 설명 추가 필요. 예: \\"원시 데이터 요약 테이블은 각 시험 조건별 평균 용량, 최대/최소 온도, 평균 전압 등의 핵심 지표를 포함하고 있으며, 이는 

In [15]:
import json
analysis_result_json = json.loads(analysis_result)

In [16]:
doc.add_paragraph("요약")
doc.add_paragraph(analysis_result_json['summary'])
doc.save("./Result/battery_report_output.docx")
print("Saved report with summary to ./Result/battery_report_output.docx")

Saved report with summary to ./Result/battery_report_output.docx


In [ ]:
## 종합 판정 추가. 
replacements = {
    "{{종합판정}}": analysis_result_json['conclusion'],
}
# docx 치환
replace_keywords_in_docx(
    input_path="./Result/battery_report_output.docx",
    output_path="./Result/battery_report_output.docx",
    replacements=replacements,
)

[완료] './Result/battery_report_output.docx' 저장 — 총 1개 단락 교체됨


1

## (6) 함수로 만들어서 리팩토링

In [27]:
## 템플릿에 넣을 변수를 추출하는 코드를 모두 작성했다면, 함수로 만들어서 재사용할 수 있도록 리팩토링
## df 입력 시 자동으로 치환된 레포트를 저장하는 함수로 만들기
## output_path는 keyword argument로, df는 positional argument로

def create_battery_report(df, output_path="./Result/battery_report_output.docx"):
    # 1. DateTime 컬럼 만들기
    df["DateTime"] = pd.to_datetime(df["Date"] + " " + df["Time"])

    # 2. 그래프 저장 폴더
    output_dir = "Result"

    # 6. 분석 텍스트 만들기
    summary_text = (
        f"본 보고서는 배터리 팩 {df['SerialNumber'].iloc[0]}의 충방전 데이터를 분석한 결과이다. "
        f"평가 기간은 {df['DateTime'].min()}부터 {df['DateTime'].max()}까지이며, "
        f"총 {len(df)}개의 데이터를 사용하였다."
    )

    capacity_text = (
        f"RSOCavg 범위는 {df['RSOCavg'].min():.2f}% ~ {df['RSOCavg'].max():.2f}%이고, "
        f"USOCavg 범위는 {df['USOCavg'].min():.2f}% ~ {df['USOCavg'].max():.2f}%이다. "
        f"SOH 평균은 {df['SOH'].mean():.2f}%이며, "
        f"Power 범위는 {df['Power'].min():.2f} ~ {df['Power'].max():.2f}이다."
    )

    welding_text = (
        f"셀 전압 최소값은 {df['Vmin'].min():.3f}V, 최대값은 {df['Vmax'].max():.3f}V이다. "
        f"셀 전압 편차 DV의 평균은 {df['DV'].mean():.2f}mV이고, "
        f"최대값은 {df['DV'].max():.2f}mV이다."
    )

    sensor_text = (
        f"온도 최소값은 {df['Tmin'].min():.2f}℃, 최대값은 {df['Tmax'].max():.2f}℃이다. "
        f"평균 온도는 {df['Tavg'].mean():.2f}℃로 확인되었다."
    )

    remark_text = (
        f"전압 범위는 {df['Voltage'].min():.2f}V ~ {df['Voltage'].max():.2f}V, "
        f"전류 범위는 {df['Current'].min():.2f}A ~ {df['Current'].max():.2f}A, "
        f"온도 범위는 {df['Tavg'].min():.2f}℃ ~ {df['Tavg'].max():.2f}℃이다."
    )


    # 7. 치환 딕셔너리 만들기
    replacements = {
        "{{작성자}}": "홍길동",
        "{{작성일}}": pd.Timestamp.now().strftime("%Y-%m-%d"),
        "{{Pack ID}}": str(df["SerialNumber"].iloc[0]),
        "{{평가기간}}": f"{df['DateTime'].min()} ~ {df['DateTime'].max()}",
        "{{AI Model}}": "Gemini-2.5-flash-lite",
        "{{요약문}}": summary_text,

        "{{용량그래프}}": {
            "type": "image",
            "stream": make_graph_stream(df, "Capacity Graph"),
            "width": 5.8,
        },
        "{{용량분석}}": capacity_text,

        "{{용접그래프}}": {
            "type": "image",
            "stream": make_graph_stream(df, "Voltage Graph"),
            "width": 5.8,
        },
        "{{용접분석}}": welding_text,

        "{{센서그래프}}": {
            "type": "image",
            "stream": make_graph_stream(df, "Temperature Graph"),
            "width": 5.8,
        },
        "{{센서분석}}": sensor_text,

        "{{샘플수}}": str(len(df)),
        "{{전압범위}}": f"{df['Voltage'].min():.2f} ~ {df['Voltage'].max():.2f} V",
        "{{전류범위}}": f"{df['Current'].min():.2f} ~ {df['Current'].max():.2f} A",
        "{{온도범위}}": f"{df['Tavg'].min():.2f} ~ {df['Tavg'].max():.2f} ℃",
        "{{비고}}": remark_text,
    }


    # 8. docx 치환
    replace_keywords_in_docx(
        input_path="./Data/Template/battery_report_template.docx",
        output_path=output_path,
        replacements=replacements,
    )
    # .docx 파일 읽기
    doc = Document(output_path)

    # 내용 추출하여 문자열로 할당
    content = ""
    for paragraph in doc.paragraphs:
        content += paragraph.text + "\n"

    # content 변수를 LLM에 입력하여 분석 결과 생성
    # 출력 형태를 json schema 로 고정하여 분석 결과를 구조화된 형태로 받도록 유도
    prompt = f"""
        다음 배터리 보고서 내용을 전반적으로 분석하고 주요 인사이트 및 결론을 요약하고, 이 평가 결과의 종합 판정을 Pass/Fail로 결정해주세요: {content}
        출력 형식은 반드시 json 형태로 {{'summary': '...', 'conclusion': 'Pass' or 'Fail'}}
    """
    analysis_result = invoke_llm(prompt)
    print(analysis_result)    

    analysis_result = analysis_result.replace("```json", "").replace("```", "")
    analysis_result_json = json.loads(analysis_result)

    doc.add_paragraph("요약")
    doc.add_paragraph(analysis_result_json['summary'])
    doc.save(output_path)
    print(f"Saved report with summary to {output_path}")

    

    ## 종합 판정 추가. 
    replacements = {
        "{{종합판정}}": analysis_result_json['conclusion'],
    }
    # docx 치환
    replace_keywords_in_docx(
        input_path=output_path,
        output_path=output_path,
        replacements=replacements,
    )

In [28]:
## 모든 데이터 레포트 생성

for data in data_list:
    data_num = data.split("_")[0]
    df = pd.read_csv(f"Data/Tabular/KAMP_Battery/{data}")
    create_battery_report(df, output_path=f"./Result/battery_report_{data_num}.docx")

/tmp/ipykernel_32637/711504412.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["DateTime"] = pd.to_datetime(df["Date"] + " " + df["Time"])


[완료] './Result/battery_report_1000.docx' 저장 — 총 17개 단락 교체됨
```json
{
  "summary": "본 배터리 충방전 시험 데이터 분석 레포트는 배터리 용량, 용접, 센서 성능을 평가합니다. \n\n**1. 용량:** 그래프 및 분석 내용은 배터리의 충방전 용량 변화를 보여주며, 전반적인 용량 유지율 및 성능 추세를 파악하는 데 사용됩니다.  \n\n**2. 용접:** 용접부의 상태를 평가하기 위한 데이터를 포함합니다. 그래프와 분석 내용은 용접 불량 여부, 강도 등을 판단하는 데 활용됩니다.\n\n**3. 센서:** 배터리 내부 센서의 측정값과 정상 범위를 비교하여 센서의 정확성 및 이상 유무를 검증합니다. \n\n**4. 원시 데이터 요약:** 시험 과정에서 수집된 모든 원시 데이터를 요약하여 제공함으로써, 세부적인 데이터 분석 및 추가 검토를 용이하게 합니다.\n\n전반적으로, 이 보고서는 배터리의 전기적 성능, 물리적 결합 상태, 내부 감지 기능 등을 종합적으로 검토하여 배터리의 품질과 신뢰성을 평가하는 데 목적이 있습니다.",
  "conclusion": "Fail"
}
```
Saved report with summary to ./Result/battery_report_1000.docx
[완료] './Result/battery_report_1000.docx' 저장 — 총 1개 단락 교체됨


/tmp/ipykernel_32637/711504412.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["DateTime"] = pd.to_datetime(df["Date"] + " " + df["Time"])


[완료] './Result/battery_report_1002.docx' 저장 — 총 17개 단락 교체됨
```json
{
  "summary": "이 보고서는 배터리 충방전 시험 데이터를 분석한 결과입니다. 주요 평가 항목은 용량, 용접, 센서이며, 각 항목별로 그래프와 분석 내용이 제공됩니다. 원시 데이터 요약도 포함되어 있어 상세한 시험 결과를 확인할 수 있습니다.\n\n**주요 인사이트:**\n\n*   **용량:** 충방전 시험을 통해 배터리의 실제 용량이 설계 기준을 만족하는지 평가합니다. 그래프는 시간 경과에 따른 충방전 용량 변화를 보여줄 것이며, 분석 내용은 기준 대비 용량의 편차, 용량 감소 추세 등을 다룰 것입니다.\n*   **용접:** 배터리 내부 셀 또는 외부 단자와의 용접 상태는 전기적 성능과 안정성에 직결됩니다. 그래프나 분석 내용은 용접 불량으로 인한 저항 증가, 이상 열 발생 가능성 등을 시사할 수 있습니다.\n*   **센서:** 배터리 관리 시스템(BMS)에 사용되는 온도, 전압 등 센서의 측정값이 정상적으로 작동하는지 평가합니다. 그래프와 분석은 센서의 정확도, 응답 속도, 이상 값 발생 여부 등을 검토할 것입니다.\n*   **원시 데이터 요약:** 시험 과정에서 수집된 모든 원시 데이터의 통계적 요약을 제공하여, 특정 구간에서의 이상 징후나 추가적인 분석을 위한 기초 자료로 활용될 수 있습니다.\n\n**전반적인 평가:**\n\n각 항목별 분석 내용과 원시 데이터 요약을 종합적으로 검토하여 배터리의 성능, 신뢰성, 안전성을 판단합니다. 특히, 용량의 기준 충족 여부, 용접 불량으로 인한 잠재적 위험, 센서의 정확한 작동 여부가 중요하게 고려될 것입니다.",
  "conclusion": "Pass"
}
```
Saved report with summary to ./Result/battery_report_1002.docx
[완료] './Result/battery_report_1002.docx' 저장 — 총 1개 단락 교체됨

/tmp/ipykernel_32637/711504412.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["DateTime"] = pd.to_datetime(df["Date"] + " " + df["Time"])


...

<Figure size 600x400 with 0 Axes>

<Figure size 600x400 with 0 Axes>

<Figure size 600x400 with 0 Axes>

<Figure size 600x400 with 0 Axes>

<Figure size 600x400 with 0 Axes>